In [1]:
include("CRD_STA.jl")
using Plots
using LinearAlgebra
using NonlinearEigenproblems
using ProgressMeter
using DelimitedFiles
using PyCall

┌ Warning: attempting to remove probably stale pidfile
│   path = "/home/zhj/.julia/compiled/v1.11/BoundaryValueDiffEqShooting/q74Uy_hcmIF.ji.pidfile"
└ @ FileWatching.Pidfile ~/.julia/juliaup/julia-1.11.5+0.x64.linux.gnu/share/julia/stdlib/v1.11/FileWatching/src/pidfile.jl:249
┌ Warning: attempting to remove probably stale pidfile
│   path = "/home/zhj/.julia/compiled/v1.11/OrdinaryDiffEq/DlSvy_hcmIF.ji.pidfile"
└ @ FileWatching.Pidfile ~/.julia/juliaup/julia-1.11.5+0.x64.linux.gnu/share/julia/stdlib/v1.11/FileWatching/src/pidfile.jl:249
┌ Warning: attempting to remove probably stale pidfile
│   path = "/home/zhj/.julia/compiled/v1.11/OrdinaryDiffEqStabilizedIRK/Ioqfb_hcmIF.ji.pidfile"
└ @ FileWatching.Pidfile ~/.julia/juliaup/julia-1.11.5+0.x64.linux.gnu/share/julia/stdlib/v1.11/FileWatching/src/pidfile.jl:249


In [ ]:
for Tw = 0.8 : 0.1 : 1.2
    for omega = -0.1 : -0.01 : -0.15
        original = read("output_GR_data_$omega _$Tw _0.3.dat",String)
        header = "Zone T = \"Tw=$Tw, Ma=0.6, omega=$omega\""
        open("output_GR_data_$omega _$Tw _0.6.dat", "w") do io
            write(io,header * "\n" * original)
        end
    end
end

In [3]:
for omega = -0.024

    for Tw = 1.2
        N_cheb = 99
        Mr = 0.3
        gamma = 1.4
        sigma = 0.72
        be_end = 0.2
        be_step = 0.001
        R_end = 100
        R_step = 0.1
        be_start = 0
        Ro = 1
        Co = 0
        Ma = Mr/R_end
        global u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,Co)
        H,T = T_ca(Mr,f,q,w0,gamma,Tw)
        F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
        lam = - (2/3) * T
        kappa = (1/sigma) * T
        global total_i_2 = []
        global total_r_2 = []
        global GR = 0
        global initial_i= []
        global initial_r= []
        global tempvec_i = []
        global tempvec_r = []
        global guess = 0
        writedlm("output_eig.dat",initial_r)
        writedlm("output_GR_$omega _$Tw _$Mr.dat",initial_r)
        writedlm("output_GR_data_$omega _$Tw _$Mr.dat",initial_r)
        for be = be_start : 1 * be_step : be_end
            A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R_end,Ma,omega,be,N_cheb,Ro,Co)
            nep = PEP([A0,A1,A2]); 
            eigval,eigvec1 = iar(nep,σ = 0.05 , neigs = 1 ,maxit = 500,tol=1e-10)
            eigval = conj(eigval[1])
            global tempvec_i = [tempvec_i;imag(eigval[1])]
            global tempvec_r = [tempvec_r;real(eigval[1])]
            open("output_eig.dat", "a") do io
                write(io,"R = $R_end,be = $be,eig = $eigval,1\n")
            end
            if length(tempvec_i) > 3 && tempvec_i[end - 1, 1] * tempvec_i[end , 1] < 0 
                initial_i = [omega R_end be imag(eigval[1])]
                initial_r = [omega R_end be real(eigval[1])]
                global beta = initial_i[end,3]
                global guess = initial_r[end,4] - im * initial_i[end,4]
                global tempvec_i = []
                global tempvec_r = []
                break
            end
        end
        for be = beta : 1 * be_step : be_end
            A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R_end,Ma,omega,be,N_cheb,Ro,Co)
            nep = PEP([A0,A1,A2]); 
            eigval,eigvec1 = iar(nep,σ = guess , neigs = 1 ,maxit = 500,tol=1e-10)
            eigval = conj(eigval[1])
            global tempvec_i = [tempvec_i;imag(eigval[1])]
            global tempvec_r = [tempvec_r;real(eigval[1])]
            if  length(tempvec_i) > 10 && tempvec_r[end - 1 ,1] > tempvec_r[end ,1] && tempvec_i[end, 1] < 0
                global type1max = tempvec_i[end - 1,1]
                global be1 = be

                break
            else
                global guess = tempvec_r[end ,1] - im * tempvec_i[end, 1]
            end
            open("output_eig.dat", "a") do io
                write(io,"R = $R_end,be = $be,eig = $eigval,2\n")
            end
        end
            global tempvec_i = [0]
            global tempvec_r = [0.7]
        for be = 0.2  : -1 * be_step : -0.1
            guess = tempvec_r[end,1] - im * tempvec_i[end, 1]
            A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R_end,Ma,omega,be,N_cheb,Ro,Co)
            nep = PEP([A0,A1,A2]); 
            eigval,eigvec1 = iar(nep,σ = guess , neigs = 1 ,maxit = 500,tol=1e-10)
            eigval = conj(eigval[1])
            global tempvec_i = [tempvec_i;imag(eigval[1])]
            global tempvec_r = [tempvec_r;real(eigval[1])]
            open("output_eig.dat", "a") do io
                write(io,"R = $R_end,be = $be,eig = $eigval,3\n")
            end
            if  length(tempvec_i) > 10 && tempvec_r[end - 1 ,1] < tempvec_r[end ,1] && tempvec_i[end, 1] < 0
                global type2max = tempvec_i[end - 1,1]
                global be2 = be
                break
            end
        end
        if  type2max < type1max
            global beta = be2
        else
            global beta = be2
        end
    end
end
#         for R = R_start  : R_step : R_end
#             Ma = Mr/R
#             A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,beta,N_cheb,1,0)
#             nep = PEP([A0,A1,A2]); 
#             eigval_2,eigvec = iar(nep , σ = 0.4 - im * total_i[end,4] , neigs = 2 ,maxit = 500,tol=1e-10)
#             eigval_2 = conj(eigval_2)
#             index = findmin(imag(eigval_2))
#             GR_temp = imag(eigval_2[index[2]]) * R_step
#             global GR = GR + abs(GR_temp)
#             open("output_GR_$omega _$Tw _$Mr.dat", "a") do io
#             write(io,"R = $R , eigmode2 = $eigval_2 , GR = $GR\n")
#             end
#             open("output_GR_data_$omega _$Tw _$Mr.dat", "a") do io
#             write(io,"$beta\t$R\t$GR\n")
#             end
#             total_i = [omega R beta imag(eigval_2[index[2]])]
#             total_r = [omega R beta real(eigval_2[index[2]])]
#         end
#     end
# end


┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBas

InterruptException: InterruptException: